In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# =========================
# PATHS
# =========================

# --- SVD ---
SVD_BASE = "/home/moshtasa/Research/phd-recsys/Movie /current/0519/SVD_0519/result"
SVD_BI_DIR = Path(SVD_BASE) / "bi"
SVD_MU_DIR = Path(SVD_BASE) / "mio"
SVD_BU_DIR = Path(SVD_BASE) / "bu"
SVD_FIG_DIR = Path(SVD_BASE) / "0519_SVD_details_figures"

# --- NMF ---
NMF_BASE = "/home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result"
NMF_BI_DIR = Path(NMF_BASE) / "bi"
NMF_MU_DIR = Path(NMF_BASE) / "mio"
NMF_BU_DIR = Path(NMF_BASE) / "bu"
NMF_FIG_DIR = Path(NMF_BASE) / "0519_NMF_details_figures"

SVD_FIG_DIR.mkdir(parents=True, exist_ok=True)
NMF_FIG_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# STYLE
# =========================

plt.style.use("seaborn-v0_8-pastel")

TARGET_COLOR = "#5B3A29"
TARGET_LINEWIDTH = 3.5
NON_TARGET_LINEWIDTH = 1.5

PASTEL_COLORS = [
    "#AEC6CF", "#FFD1DC", "#B5EAD7", "#FFDAC1", "#C3B1E1",
    "#F3E5AB", "#FFB7B2", "#B0E0E6", "#D5AAFF", "#FFFACD",
    "#E0BBE4", "#957DAD", "#D4E157", "#80CBC4", "#F48FB1",
]

TARGET_BIAS_COLOR = "#8B0000"
TARGET_NONBIAS_COLOR = "#5B3A29"
NONTARGET_BIAS_COLOR = "#1F77B4"
NONTARGET_NONBIAS_COLOR = "#7FB3D5"

# =========================
# HELPERS
# =========================

def clean_df(df):
    if "decade" in df.columns:
        df["decade"] = df["decade"].astype(str).str.strip()
        df = df[df["decade"].str.lower() != "unknown"].copy()

    if "is_target" in df.columns:
        df["is_target"] = df["is_target"].astype(str).str.strip()

    if "user_type" in df.columns:
        df["user_type"] = df["user_type"].astype(str).str.strip()

    return df


def get_single_value(row, col):
    if row.empty or col not in row.columns:
        return None
    return row[col].values[0]


def get_output_dirs(decade_key):
    svd_dir = SVD_FIG_DIR / str(decade_key)
    nmf_dir = NMF_FIG_DIR / str(decade_key)

    svd_dir.mkdir(parents=True, exist_ok=True)
    nmf_dir.mkdir(parents=True, exist_ok=True)

    return svd_dir, nmf_dir


def load_and_merge(bi_dir, mu_dir, bu_dir, bi_pattern):

    bi_files = {}
    merged_data = {}
    bu_data = {}

    for fname in os.listdir(bi_dir):
        m = bi_pattern.match(fname)
        if m:
            decade_key = m.group(1)
            n = int(m.group(2))
            bi_files[(decade_key, n)] = fname

    print(f"Found {len(bi_files)} bi files in {bi_dir}")

    for (decade_key, n), bi_fname in sorted(bi_files.items()):

        mu_fname = bi_fname.replace("_bi_", "_mu_")

        bu_fname = (
            bi_fname
            .replace("_bi_", "_bu_")
            .replace("_per_decade.csv", "_per_decade_biased.csv")
        )

        bi_path = Path(bi_dir) / bi_fname
        mu_path = Path(mu_dir) / mu_fname
        bu_path = Path(bu_dir) / bu_fname

        if not (mu_path.exists() and bu_path.exists()):
            print(f"WARNING: Missing mu/bu for {bi_fname}, skipping.")
            continue

        df_bi = clean_df(pd.read_csv(bi_path))
        df_mu = clean_df(pd.read_csv(mu_path))
        df_bu = clean_df(pd.read_csv(bu_path))

        bu_data[(decade_key, n)] = df_bu.copy()

        if "user_type" in df_bu.columns:
            df_bu_merge = df_bu[df_bu["user_type"] == "non_biasing"].copy()
            if df_bu_merge.empty:
                df_bu_merge = df_bu.copy()
        else:
            df_bu_merge = df_bu.copy()

        merge_cols = ["decade"]
        if "is_target" in df_bi.columns:
            merge_cols.append("is_target")

        df = pd.merge(df_mu, df_bi, on=merge_cols, how="outer")
        df = pd.merge(df, df_bu_merge, on=merge_cols, how="outer")

        merged_data[(decade_key, n)] = df

    print(f"Merged {len(merged_data)} configs")

    return merged_data, bu_data


def make_groups(data):
    groups = defaultdict(dict)
    for (k, n), df in data.items():
        groups[k][n] = df
    return groups


# =========================
# LOAD DATA
# =========================

BI_PATTERN = re.compile(
    r"^f_(.+?)_(\d+)u_pos5_neg1_movies_bi_per_decade\.csv$"
)

svd_data, svd_bu_data = load_and_merge(
    SVD_BI_DIR,
    SVD_MU_DIR,
    SVD_BU_DIR,
    BI_PATTERN
)

nmf_data, nmf_bu_data = load_and_merge(
    NMF_BI_DIR,
    NMF_MU_DIR,
    NMF_BU_DIR,
    BI_PATTERN
)

svd_groups = make_groups(svd_data)
nmf_groups = make_groups(nmf_data)

svd_bu_groups = make_groups(svd_bu_data)
nmf_bu_groups = make_groups(nmf_bu_data)

# =========================
# SIDE-BY-SIDE METRIC FIGURES
# =========================

def plot_side_by_side(
    decade_key,
    nmf_dict,
    svd_dict,
    sorted_ns,
    metric_name,
    ylabel,
    suffix,
    compute_value
):

    sample_df = nmf_dict[sorted_ns[0]]

    if "is_target" not in sample_df.columns:
        return

    target_rows = sample_df[sample_df["is_target"] == "target"]

    if target_rows.empty:
        return

    target_decade_name = target_rows["decade"].iloc[0]

    def build_series(n_dict):

        all_decades = set()

        for n in sorted_ns:
            all_decades.update(
                n_dict[n]["decade"]
                .dropna()
                .tolist()
            )

        all_decades = sorted(all_decades)

        series = {}

        for decade_name in all_decades:

            pts = []

            for n in sorted_ns:

                df = n_dict[n]
                row = df[df["decade"] == decade_name]

                if not row.empty:

                    val = compute_value(row)

                    if val is not None:
                        pts.append((n, val))

            series[decade_name] = pts

        return series, all_decades

    nmf_series, all_decades = build_series(nmf_dict)
    svd_series, _ = build_series(svd_dict)

    non_target = [
        g for g in all_decades
        if g != target_decade_name
    ]

    color_map = {
        g: PASTEL_COLORS[i % len(PASTEL_COLORS)]
        for i, g in enumerate(non_target)
    }

    color_map[target_decade_name] = TARGET_COLOR

    positions = list(range(len(sorted_ns)))

    n_to_pos = {
        n: p
        for n, p in zip(sorted_ns, positions)
    }

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(16, 6),
        sharey=True
    )

    legend_handles = {}

    for ax, series, model_name in zip(
        axes,
        [nmf_series, svd_series],
        ["NMF", "SVD"]
    ):

        for g in non_target:

            pts = series.get(g, [])

            if pts:

                xs = [n_to_pos[x] for x, _ in pts]
                ys = [y for _, y in pts]

                line, = ax.plot(
                    xs,
                    ys,
                    marker="o",
                    color=color_map[g],
                    linewidth=NON_TARGET_LINEWIDTH,
                    markersize=9,
                    alpha=0.8,
                    label=g
                )

                if g not in legend_handles:
                    legend_handles[g] = line

        pts = series.get(target_decade_name, [])

        if pts:

            xs = [n_to_pos[x] for x, _ in pts]
            ys = [y for _, y in pts]

            line, = ax.plot(
                xs,
                ys,
                marker="o",
                color=TARGET_COLOR,
                linewidth=TARGET_LINEWIDTH,
                markersize=7,
                label=f"{target_decade_name} (target)",
                zorder=10
            )

            legend_handles[target_decade_name] = line

        ax.tick_params(axis='y', labelsize=18)
        ax.tick_params(axis='x', labelsize=20)

        ax.set_xlabel(
            "Number of Fictitious Users (n)",
            fontsize=22,
            fontweight="bold"
        )

        ax.set_xticks(positions)

        ax.set_xticklabels(
            [str(n) for n in sorted_ns],
            rotation=45
        )

        ax.grid(True, alpha=0.3)

        ax.set_title(
            f"{model_name} - Decade {decade_key})",
            fontsize=22,
            fontweight="bold"
        )

    axes[0].set_ylabel(
        ylabel,
        fontweight="bold",
        fontsize=26
    )

    fig.legend(
        list(legend_handles.values()),
        list(legend_handles.keys()),
        loc="upper center",
        bbox_to_anchor=(0.5, 1.1),
        fontsize=16,
        ncol=min(len(legend_handles), 6),
        handlelength=1.2,
        handletextpad=0.6,
        columnspacing=0.8,
        frameon=True
    )

    plt.tight_layout(rect=[0, 0, 1, 0.92])

    filename = f"{suffix}_comparison.png"

    svd_dir, nmf_dir = get_output_dirs(decade_key)

    svd_path = svd_dir / filename
    nmf_path = nmf_dir / filename

    fig.savefig(svd_path, dpi=150, bbox_inches="tight")
    fig.savefig(nmf_path, dpi=150, bbox_inches="tight")

    plt.close(fig)

    print(f"Saved: {svd_path}")
    print(f"Saved: {nmf_path}")


# =========================
# SIDE-BY-SIDE BU BIASED VS UNBIASED
# =========================

def build_bu_bias_series(n_dict, sorted_ns):

    target_bias_pts = []
    target_nonbias_pts = []
    non_target_bias_pts = []
    non_target_nonbias_pts = []

    for n in sorted_ns:

        df = n_dict[n].copy()

        if "decade" in df.columns:
            df["decade"] = df["decade"].astype(str).str.strip()
            df = df[df["decade"].str.lower() != "unknown"]

        if "user_type" in df.columns:
            df["user_type"] = df["user_type"].astype(str).str.strip()

        if "is_target" in df.columns:
            df["is_target"] = df["is_target"].astype(str).str.strip()

        row = df[
            (df["is_target"] == "target")
            & (df["user_type"] == "biasing")
        ]

        if not row.empty:
            target_bias_pts.append((n, row["avg_bu"].mean()))

        row = df[
            (df["is_target"] == "target")
            & (df["user_type"] == "non_biasing")
        ]

        if not row.empty:
            target_nonbias_pts.append((n, row["avg_bu"].mean()))

        row = df[
            (df["is_target"] == "non_target")
            & (df["user_type"] == "biasing")
        ]

        if not row.empty:
            non_target_bias_pts.append((n, row["avg_bu"].mean()))

        row = df[
            (df["is_target"] == "non_target")
            & (df["user_type"] == "non_biasing")
        ]

        if not row.empty:
            non_target_nonbias_pts.append((n, row["avg_bu"].mean()))

    return {
        "Target - biasing": target_bias_pts,
        "Target - non_biasing": target_nonbias_pts,
        "Non-target - biasing": non_target_bias_pts,
        "Non-target - non_biasing": non_target_nonbias_pts,
    }


def plot_bu_bias_side_by_side(
    decade_key,
    nmf_dict,
    svd_dict,
    sorted_ns
):

    nmf_series = build_bu_bias_series(nmf_dict, sorted_ns)
    svd_series = build_bu_bias_series(svd_dict, sorted_ns)

    positions = list(range(len(sorted_ns)))

    n_to_pos = {
        n: p
        for n, p in zip(sorted_ns, positions)
    }

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(16, 6),
        sharey=True
    )

    series_style = {
        "Target - biasing": (TARGET_BIAS_COLOR, 3.5, "o"),
        "Target - non_biasing": (TARGET_NONBIAS_COLOR, 3.5, "s"),
        "Non-target - biasing": (NONTARGET_BIAS_COLOR, 2.5, "^"),
        "Non-target - non_biasing": (NONTARGET_NONBIAS_COLOR, 2.5, "D"),
    }

    legend_handles = {}

    for ax, series, model_name in zip(
        axes,
        [nmf_series, svd_series],
        ["NMF", "SVD"]
    ):

        for label, pts in series.items():

            if not pts:
                continue

            color, linewidth, marker = series_style[label]

            xs = [n_to_pos[x] for x, _ in pts]
            ys = [y for _, y in pts]

            line, = ax.plot(
                xs,
                ys,
                marker=marker,
                color=color,
                linewidth=linewidth,
                markersize=8,
                label=label
            )

            if label not in legend_handles:
                legend_handles[label] = line

        ax.tick_params(axis='y', labelsize=18)
        ax.tick_params(axis='x', labelsize=20)

        ax.set_xlabel(
            "Number of Fictitious Users (n)",
            fontsize=22,
            fontweight="bold"
        )

        ax.set_xticks(positions)

        ax.set_xticklabels(
            [str(n) for n in sorted_ns],
            rotation=45
        )

        ax.grid(True, alpha=0.3)

        ax.set_title(
            f"{model_name} - Decade {decade_key}",
            fontsize=22,
            fontweight="bold"
        )

    axes[0].set_ylabel(
        r"$b_u$",
        fontweight="bold",
        fontsize=26
    )

    fig.legend(
        list(legend_handles.values()),
        list(legend_handles.keys()),
        loc="upper center",
        bbox_to_anchor=(0.5, 1),
        fontsize=18,
        ncol=4,
        handlelength=1.2,
        handletextpad=0.6,
        columnspacing=0.8,
        frameon=True
    )

    plt.tight_layout(rect=[0, 0, 1, 0.92])

    filename = "bu_biased_vs_unbiased_target_vs_nontarget_comparison.png"

    svd_dir, nmf_dir = get_output_dirs(decade_key)

    svd_path = svd_dir / filename
    nmf_path = nmf_dir / filename

    fig.savefig(svd_path, dpi=150, bbox_inches="tight")
    fig.savefig(nmf_path, dpi=150, bbox_inches="tight")

    plt.close(fig)

    print(f"Saved: {svd_path}")
    print(f"Saved: {nmf_path}")


# =========================
# MAIN LOOP
# =========================

for decade_key in sorted(nmf_groups.keys()):

    if decade_key not in svd_groups:
        continue

    nmf_dict = nmf_groups[decade_key]
    svd_dict = svd_groups[decade_key]

    sorted_ns = sorted(
        set(nmf_dict.keys())
        & set(svd_dict.keys())
    )

    if not sorted_ns:
        continue

    plot_side_by_side(
        decade_key,
        nmf_dict,
        svd_dict,
        sorted_ns,
        "μ + bᵢ",
        r"$\mu + b_i$",
        "mu_plus_bi",
        lambda r: (
            get_single_value(r, "global_mean_mu")
            + get_single_value(r, "avg_bi")
            if get_single_value(r, "global_mean_mu") is not None
            and get_single_value(r, "avg_bi") is not None
            else None
        )
    )

    plot_side_by_side(
        decade_key,
        nmf_dict,
        svd_dict,
        sorted_ns,
        "bᵢ",
        r"$b_i$",
        "bi_only",
        lambda r: get_single_value(r, "avg_bi")
    )

    plot_side_by_side(
        decade_key,
        nmf_dict,
        svd_dict,
        sorted_ns,
        "bᵤ (non-biasing users)",
        r"$b_u$",
        "bu_only_non_biasing",
        lambda r: get_single_value(r, "avg_bu")
    )

    plot_side_by_side(
        decade_key,
        nmf_dict,
        svd_dict,
        sorted_ns,
        "μ",
        r"$\mu$",
        "mu_only",
        lambda r: get_single_value(r, "global_mean_mu")
    )

    if decade_key in nmf_bu_groups and decade_key in svd_bu_groups:

        nmf_bu_dict = nmf_bu_groups[decade_key]
        svd_bu_dict = svd_bu_groups[decade_key]

        bu_sorted_ns = sorted(
            set(nmf_bu_dict.keys())
            & set(svd_bu_dict.keys())
        )

        if bu_sorted_ns:

            plot_bu_bias_side_by_side(
                decade_key,
                nmf_bu_dict,
                svd_bu_dict,
                bu_sorted_ns
            )

print("Done!")

Found 32 bi files in /home/moshtasa/Research/phd-recsys/Movie /current/0519/SVD_0519/result/bi
Merged 32 configs
Found 32 bi files in /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/bi
Merged 32 configs
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/SVD_0519/result/0519_SVD_details_figures/1920/mu_plus_bi_comparison.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_details_figures/1920/mu_plus_bi_comparison.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/SVD_0519/result/0519_SVD_details_figures/1920/bi_only_comparison.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_details_figures/1920/bi_only_comparison.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/SVD_0519/result/0519_SVD_details_figures/1920/bu_only_non_biasing_comparison.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_details_figure